# Outcome pairwise — McNemar's exact

Which backend *pairs* differ on outcomes, with Bonferroni correction.

In [ ]:
from analysis import load_runs

# Pin matplotlib (Agg) + RNG seeds for reproducible, headless figures.
plt = load_runs.pin_style(seed=0)

# Load real runs if any exist, else the deterministic synthetic dataset.
records = load_runs.load_all()
backends = load_runs.backend_names(records)
STRATEGY = "zero_shot"  # the held-fixed strategy for the cross-backend tests
print(f"{len(records)} runs, backends={backends}")

In [ ]:
import numpy as np

from analysis import stats as st
from analysis.aggregate import aggregate_runs, build_outcome_matrix

summaries = aggregate_runs(records)
binary_ids, matrix = build_outcome_matrix(summaries, backends=backends, strategy=STRATEGY)
cols = {b: [row[j] for row in matrix] for j, b in enumerate(backends)}
pairs = st.pairwise_labels(backends)
alpha = st.bonferroni_alpha(0.05, len(pairs))
results = [st.mcnemar_exact(cols[a], cols[b], label_a=a, label_b=b) for a, b in pairs]
for r in results:
    sig = "*" if r.p_value < alpha else " "
    print(
        f"{sig} {r.backend_a} vs {r.backend_b}: b={r.b} c={r.c} "
        f"p={r.p_value:.4g} g={r.cohens_g:+.3f}"
    )
print(f"Bonferroni-corrected alpha = {alpha:.4f}")

In [ ]:
k = len(backends)
pmat = np.ones((k, k))
for r in results:
    i, j = backends.index(r.backend_a), backends.index(r.backend_b)
    pmat[i, j] = pmat[j, i] = r.p_value
fig, ax = plt.subplots()
im = ax.imshow(pmat, vmin=0, vmax=1, cmap="viridis_r")
ax.set_xticks(range(k))
ax.set_xticklabels(backends, rotation=20, ha="right")
ax.set_yticks(range(k))
ax.set_yticklabels(backends)
for i in range(k):
    for j in range(k):
        ax.text(j, i, f"{pmat[i, j]:.2g}", ha="center", va="center", color="white", fontsize=8)
ax.set_title("McNemar exact p-values (pairwise)")
fig.colorbar(im, ax=ax, label="p-value")
load_runs.save_figure(fig, "02_outcome_pairwise", "mcnemar_pvalues")
fig

In [ ]:
from IPython.display import Markdown

lines = [
    f"| {r.backend_a} | {r.backend_b} | {r.b} | {r.c} | {r.p_value:.4g} | "
    f"{r.cohens_g:+.3f} | {'yes' if r.p_value < alpha else 'no'} |"
    for r in results
]
Markdown(
    "### Summary — outcome pairwise (McNemar exact, Bonferroni)\n\n"
    f"Corrected alpha = **{alpha:.4f}** over {len(pairs)} comparisons.\n\n"
    "| A | B | b | c | p | Cohen's g | sig |\n|---|---|---|---|---|---|---|\n" + "\n".join(lines)
)